In [0]:
# ============================================================
# NOTEBOOK 03 — TARGET DATASETS + BALANCING
# Section 4.2 du papier Belcastro et al. (2016)
# ============================================================

# CELLULE 1 — Initialisation
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, rand
from pyspark.sql.types import IntegerType

spark = SparkSession.builder \
    .appName("FlightDelay_Notebook03") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

print(f"✅ Spark {spark.version} prêt !")

In [0]:
# ============================================================
# CELLULE 2 — Chargement JT
# ============================================================
df_JT = spark.read.parquet(
    "/Volumes/workspace/default/outputs/JT/"
)

n_total  = df_JT.count()
n_del_15 = df_JT.filter(col("label_15") == 1).count()
n_del_60 = df_JT.filter(col("label_60") == 1).count()

print(f"✅ JT chargée  : {n_total:,} vols")
print(f"📋 Colonnes    : {len(df_JT.columns)}")
print(f"🏷️  label_15  : {n_del_15:,} delayed ({n_del_15/n_total*100:.1f}%)")
print(f"🏷️  label_60  : {n_del_60:,} delayed ({n_del_60/n_total*100:.1f}%)")

In [0]:
# ============================================================
# CELLULE 3 — Construction D1, D2, D3, D4
# Section 4.2 du papier — Table VI
#
# D1 : delayed DUE ONLY to extreme weather OR NAS
#      (ou combinaison exclusive des deux)
#      → WEATHER_DELAY > 0 XOR NAS_DELAY > 0
#        OU (WEATHER_DELAY > 0 AND NAS_DELAY > 0)
#        ET aucune autre cause
#      ⚠️ Approximation : sans CARRIER_DELAY etc.
#         → on garde vols avec WEATHER ou NAS > 0
#            et ARR_DELAY ≈ WEATHER + NAS
#
# D2 : delayed affected by extreme weather
#      PLUS those for which NAS >= threshold
#
# D3 : delayed affected by extreme weather OR NAS
#      even if not exclusively
#
# D4 : ALL delayed flights (référence)
# ============================================================

def build_target_datasets(df_JT, threshold):
    """
    Construit D1, D2, D3, D4 selon Section 4.2 du papier.
    threshold : 15 ou 60 (minutes)
    """
    label_col = f"label_{threshold}"

    df_ontime  = df_JT.filter(col(label_col) == 0)
    df_delayed = df_JT.filter(col(label_col) == 1)

    # ── D1 : Solo Extreme ∪ Solo NAS ∪ Solo (Extreme AND NAS)
    # Approximation : WEATHER ou NAS est la seule cause visible
    D1_delayed = df_delayed.filter(
        (col("WEATHER_DELAY") > 0) | (col("NAS_DELAY") > 0)
    ).filter(
        # Pas d'autres causes connues = ARR_DELAY ≈ WEATHER + NAS
        col("ARR_DELAY_NEW") <= (
            col("WEATHER_DELAY") + col("NAS_DELAY") + 5
        )
    )

    # ── D2 : Extreme ∪ (NAS >= threshold)
    D2_delayed = df_delayed.filter(
        (col("WEATHER_DELAY") > 0) |
        (col("NAS_DELAY") >= threshold)
    )

    # ── D3 : Extreme ∪ NAS (même combiné avec autres causes)
    D3_delayed = df_delayed.filter(
        (col("WEATHER_DELAY") > 0) | (col("NAS_DELAY") > 0)
    )

    # ── D4 : Tous les delayed
    D4_delayed = df_delayed

    print(f"\n{'='*50}")
    print(f"  DATASETS — Threshold {threshold} min")
    print(f"{'='*50}")
    print(f"  OnTime total : {df_ontime.count():,}")
    print(f"  D1 delayed   : {D1_delayed.count():,}"
          f" ({D1_delayed.count()/df_delayed.count()*100:.1f}% des delayed)")
    print(f"  D2 delayed   : {D2_delayed.count():,}"
          f" ({D2_delayed.count()/df_delayed.count()*100:.1f}% des delayed)")
    print(f"  D3 delayed   : {D3_delayed.count():,}"
          f" ({D3_delayed.count()/df_delayed.count()*100:.1f}% des delayed)")
    print(f"  D4 delayed   : {D4_delayed.count():,}"
          f" ({D4_delayed.count()/df_delayed.count()*100:.1f}% des delayed)")

    return df_ontime, D1_delayed, D2_delayed, D3_delayed, D4_delayed

ontime_15, D1_15, D2_15, D3_15, D4_15 = build_target_datasets(df_JT, 15)
ontime_60, D1_60, D2_60, D3_60, D4_60 = build_target_datasets(df_JT, 60)

In [0]:
# ============================================================
# CELLULE 4 — Balancing + Train/Test Split
# Section 4.2 + Figure 4 du papier :
#
# "delayed tuples were randomly added to the training
#  and test sets with a 3:1 ratio"
# → 75% train / 25% test pour les delayed
#
# "on-time instances were randomly added, without
#  repetition, until the number of delayed and on-time
#  instances were the same"
# → under-sampling ontime pour 50/50
# ============================================================

def balance_and_split(df_ontime, df_delayed, name,
                      label_col, seed=42):
    """
    Reproduit exactement Figure 4 du papier :
    1. Split delayed → 75% train / 25% test (ratio 3:1)
    2. Under-sample ontime → 50/50 dans train ET test
    """
    n_delayed = df_delayed.count()
    n_ontime  = df_ontime.count()

    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(f"  Delayed : {n_delayed:,} | OnTime : {n_ontime:,}")

    if n_delayed == 0:
        print(f"  ⚠️  Pas de delayed — skip !")
        return None, None

    # ── Étape 1 : Split delayed 75/25 (ratio 3:1)
    df_del_train, df_del_test = df_delayed.randomSplit(
        [0.75, 0.25], seed=seed
    )
    n_del_train = df_del_train.count()
    n_del_test  = df_del_test.count()

    print(f"  Delayed train : {n_del_train:,} | test : {n_del_test:,}")

    # ── Étape 2 : Under-sample ontime
    # On veut exactement n_del_train ontime dans train
    # et n_del_test ontime dans test
    # → shuffle ontime puis prendre les N premiers

    df_ontime_shuffled = df_ontime.orderBy(rand(seed))

    df_on_train = df_ontime_shuffled.limit(n_del_train)
    df_on_test  = df_ontime_shuffled \
        .subtract(df_on_train) \
        .limit(n_del_test)

    # ── Étape 3 : Union + shuffle final
    df_train = df_del_train \
        .union(df_on_train) \
        .orderBy(rand(seed + 1))
    df_test  = df_del_test \
        .union(df_on_test) \
        .orderBy(rand(seed + 2))

    n_train     = df_train.count()
    n_test      = df_test.count()
    pct_del_tr  = df_train.filter(
        col(label_col) == 1).count() / n_train * 100
    pct_del_te  = df_test.filter(
        col(label_col) == 1).count() / n_test * 100

    print(f"  ✅ Train : {n_train:,} ({pct_del_tr:.1f}% delayed)")
    print(f"  ✅ Test  : {n_test:,}  ({pct_del_te:.1f}% delayed)")

    return df_train, df_test

In [0]:
# ============================================================
# CELLULE 5 — Appliquer sur tous les datasets
# ============================================================

configs = [
    ("D1_th15", ontime_15, D1_15, "label_15"),
    ("D2_th15", ontime_15, D2_15, "label_15"),
    ("D3_th15", ontime_15, D3_15, "label_15"),
    ("D4_th15", ontime_15, D4_15, "label_15"),
    ("D1_th60", ontime_60, D1_60, "label_60"),
    ("D2_th60", ontime_60, D2_60, "label_60"),
    ("D3_th60", ontime_60, D3_60, "label_60"),
    ("D4_th60", ontime_60, D4_60, "label_60"),
]

results = {}
for name, df_on, df_del, label in configs:
    train, test = balance_and_split(
        df_on, df_del, name, label
    )
    results[name] = (train, test)


In [0]:
# ============================================================
# CELLULE 6 — Sauvegarde
# ============================================================

OUTPUT = "/Volumes/workspace/default/outputs/"

for name, (train, test) in results.items():
    if train is None:
        continue
    train.write.mode("overwrite").parquet(
        f"{OUTPUT}{name}_train/"
    )
    test.write.mode("overwrite").parquet(
        f"{OUTPUT}{name}_test/"
    )
    print(f"✅ {name} sauvegardé !")

In [0]:
# ============================================================
# CELLULE 7 — Table VI du papier (reproduction)
# ============================================================

print("\n" + "="*65)
print("   TABLE VI — Features of target datasets (papier p.10)")
print("="*65)
print(f"{'Dataset':<12} {'#Delayed':>10} {'%Delayed':>9} "
      f"{'#Train':>10} {'#Test':>10}")
print("-"*65)

papier = {
    "D1_th15": ("22.9%", "1.3M"),
    "D2_th15": ("37.1%", "2.1M"),
    "D3_th15": ("58.9%", "3.4M"),
    "D4_th15": ("100%",  "5.8M"),
    "D1_th60": ("15.4%", "257k"),
    "D2_th60": ("25.9%", "433k"),
    "D3_th60": ("56.8%", "950k"),
    "D4_th60": ("100%",  "1.7M"),
}

for name, (train, test) in results.items():
    if train is None:
        continue
    label = "label_15" if "th15" in name else "label_60"
    n_del  = train.filter(col(label) == 1).count()
    n_tr   = train.count()
    n_te   = test.count()
    pct    = n_del / n_tr * 100
    p_pct, p_n = papier[name]
    print(f"{name:<12} {n_del:>10,} {pct:>8.1f}% "
          f"{n_tr:>10,} {n_te:>10,}  "
          f"[papier: {p_n} / {p_pct}]")

print("="*65)
print("\n✅ Notebook 03 TERMINÉ → Prêt pour Notebook 04 !")

In [0]:
import matplotlib.pyplot as plt
import numpy as np

# ── Données réelles après balancing ──
data = {
    "D1_th15": {"notre": 247_180,   "papier": 1_300_000},
    "D2_th15": {"notre": 379_296,   "papier": 2_100_000},
    "D3_th15": {"notre": 609_624,   "papier": 3_400_000},
    "D4_th15": {"notre": 1_057_008, "papier": 5_800_000},
    "D1_th60": {"notre": 52_026,    "papier": 257_000},
    "D2_th60": {"notre": 86_768,    "papier": 433_000},
    "D3_th60": {"notre": 187_228,   "papier": 950_000},
    "D4_th60": {"notre": 330_530,   "papier": 1_700_000},
}

labels  = list(data.keys())
notre   = [data[k]["notre"]  / 1_000 for k in labels]  # en milliers
papier  = [data[k]["papier"] / 1_000 for k in labels]

x     = np.arange(len(labels))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    "Volumes des datasets cibles — Notre implémentation vs Papier",
    fontsize=13, fontweight="bold"
)

for ax_idx, (threshold, indices) in enumerate(
    [("th=15 (seuil 15 min)", [0,1,2,3]),
     ("th=60 (seuil 60 min)", [4,5,6,7])]
):
    ax   = axes[ax_idx]
    lbls = [labels[i] for i in indices]
    xi   = np.arange(len(lbls))

    v_notre  = [notre[i]  for i in indices]
    v_papier = [papier[i] for i in indices]

    b1 = ax.bar(xi - width/2, v_notre,  width,
                label="Notre impl.", color="#1f77b4", alpha=0.85)
    b2 = ax.bar(xi + width/2, v_papier, width,
                label="Papier",      color="#d62728", alpha=0.85)

    # Valeurs sur les barres
    for bar in [b1, b2]:
        for rect in bar:
            h = rect.get_height()
            ax.text(
                rect.get_x() + rect.get_width()/2,
                h + 10, f"{h:.0f}k",
                ha="center", va="bottom", fontsize=7.5
            )

    # Ratio notre/papier au dessus
    for i, (n, p) in enumerate(zip(v_notre, v_papier)):
        ratio = n / p * 100
        ax.text(xi[i], max(n, p) + 60,
                f"{ratio:.0f}%",
                ha="center", fontsize=8,
                color="green", fontweight="bold")

    ax.set_title(f"Threshold {threshold}", fontsize=11)
    ax.set_xticks(xi)
    ax.set_xticklabels(lbls, rotation=15, fontsize=9)
    ax.set_ylabel("Nombre de tuples (milliers)")
    ax.legend(fontsize=9)
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    ax.spines[["top", "right"]].set_visible(False)

# Note en bas
fig.text(0.5, -0.02,
         "Pourcentages verts = ratio notre impl. / papier  "
         "— Écart dû à la couverture météo partielle (22% de FT)",
         ha="center", fontsize=8, color="gray", style="italic")

plt.tight_layout()
plt.savefig("diffe_stat.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Graphique sauvegardé → diffe_stat.png")